# Comparacao RAG Estruturado (v2)

Esta versao usa:
- **GraphRAG** (mesmo fluxo do projeto)
- **Naive RAG com OpenSearch** (modulo independente `naive-rag-opensearch`)

Inclui indexacao dos documentos no OpenSearch dentro do notebook.

A comparacao foi refatorada para usar um unico LLM configuravel nas abordagens e nas avaliacoes. Configure `BENCHMARK_LLM_PROVIDER` (`openai` ou `azure`) e `BENCHMARK_LLM_MODEL` (deployment/modelo GPT) para controlar todo o benchmark. Para Azure, defina tambem `AZURE_OPENAI_ENDPOINT`, `AZURE_OPENAI_API_KEY` e opcionalmente `AZURE_OPENAI_API_VERSION` (default `2024-10-21`).

A **avaliacao de recuperacao** usa as metricas **Context Precision** e **Context Recall** do [RAGAS](https://docs.ragas.io/) (juizo por LLM sobre a utilidade dos contextos e sobre a cobertura da resposta de referencia). No GraphRAG, os contextos incluem entidades, relacionamentos, claims/covariates e trechos (`context_docs` empacotado pelo fluxo local/global), nao apenas os TextUnits como no RAG tradicional (chunks do OpenSearch). Variaveis opcionais adicionais: `RAGAS_LLM_MODEL` (override do modelo de juizo), `RAGAS_MAX_CONTEXT_BLOCKS` (default 16; limite de blocos por pergunta para custo da metrica de precisao).

In [52]:
from __future__ import annotations

import csv
import importlib
import math
import os
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "graphrag-neo4j-langchain").is_dir():
    if (PROJECT_ROOT.parent / "graphrag-neo4j-langchain").is_dir():
        PROJECT_ROOT = PROJECT_ROOT.parent

GRAPHRAG_SRC = PROJECT_ROOT / "graphrag-neo4j-langchain" / "src"
NAIVE_OS_SRC = PROJECT_ROOT / "naive-rag-opensearch"
DOCS_DIR = (PROJECT_ROOT / "pdfs_txt").resolve()
QA_CSV = (PROJECT_ROOT / "docs" / "rag_eval_qa_estruturado.csv").resolve()

if not GRAPHRAG_SRC.is_dir():
    raise FileNotFoundError(f"Nao encontrei graphrag em: {GRAPHRAG_SRC}")
if not NAIVE_OS_SRC.is_dir():
    raise FileNotFoundError(f"Nao encontrei modulo naive-rag-opensearch em: {NAIVE_OS_SRC}")

for p in (str(GRAPHRAG_SRC), str(NAIVE_OS_SRC)):
    if p not in sys.path:
        sys.path.insert(0, p)

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"DOCS_DIR: {DOCS_DIR}")
print(f"QA_CSV: {QA_CSV}")

PROJECT_ROOT: /home/micael/mestrado/benchmarking-graphrag
DOCS_DIR: /home/micael/mestrado/benchmarking-graphrag/pdfs_txt
QA_CSV: /home/micael/mestrado/benchmarking-graphrag/docs/rag_eval_qa_estruturado.csv


In [53]:
from dotenv import load_dotenv
import socket

# Carrega .env do GraphRAG e do modulo Naive RAG OpenSearch.
load_dotenv(PROJECT_ROOT / "graphrag-neo4j-langchain" / ".env", override=False)
load_dotenv(PROJECT_ROOT / "naive-rag-opensearch" / ".env", override=False)

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY nao definida. Configure em um dos .env.")

# Ajusta host/porta do OpenSearch para funcionar tanto dentro quanto fora do
# container 'comparison'.
os_host = os.getenv("OPENSEARCH_HOST", "localhost")
os_port = os.getenv("OPENSEARCH_PORT", "9200")

# Dentro da rede docker-compose: API HTTP do OpenSearch em opensearch:9200.
if os_host == "opensearch" and os_port == "9300":
    os.environ["OPENSEARCH_PORT"] = "9200"
    os_port = "9200"

# Fora do compose, o hostname 'opensearch' nao resolve; use porta publicada.
if os_host == "opensearch":
    try:
        socket.getaddrinfo("opensearch", int(os_port))
    except OSError:
        os.environ["OPENSEARCH_HOST"] = "localhost"
        os_host = "localhost"
        if os_port == "9200":
            os.environ["OPENSEARCH_PORT"] = "9300"
            os_port = "9300"

print("OPENAI_API_KEY: OK")
print("OPENSEARCH_HOST:", os_host)
print("OPENSEARCH_PORT:", os_port)
print("OPENSEARCH_INDEX:", os.getenv("OPENSEARCH_INDEX", "naive_rag_chunks"))

OPENAI_API_KEY: OK
OPENSEARCH_HOST: localhost
OPENSEARCH_PORT: 9300
OPENSEARCH_INDEX: naive_rag_chunks


In [54]:
@dataclass
class QARow:
    qid: str
    tipo_pergunta: str
    question: str
    trecho_1: str
    fonte_trecho_1: str
    trecho_2: str
    fonte_trecho_2: str
    reference_answer: str
    ground_truth_facts_text: str


def load_qa_estruturado_csv(path: Path) -> list[QARow]:
    required = {
        "id",
        "tipo_pergunta",
        "pergunta",
        "trecho_1",
        "fonte_trecho_1",
        "trecho_2",
        "fonte_trecho_2",
        "resposta_esperada",
    }
    rows: list[QARow] = []
    with path.open("r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)
        miss = required - set(reader.fieldnames or [])
        if miss:
            raise ValueError(f"CSV incompleto. Faltam colunas: {sorted(miss)}")
        for r in reader:
            trecho_1 = (r.get("trecho_1") or "").strip()
            trecho_2 = (r.get("trecho_2") or "").strip()
            rows.append(
                QARow(
                    qid=(r.get("id") or "").strip(),
                    tipo_pergunta=(r.get("tipo_pergunta") or "").strip(),
                    question=(r.get("pergunta") or "").strip(),
                    trecho_1=trecho_1,
                    fonte_trecho_1=(r.get("fonte_trecho_1") or "").strip(),
                    trecho_2=trecho_2,
                    fonte_trecho_2=(r.get("fonte_trecho_2") or "").strip(),
                    reference_answer=(r.get("resposta_esperada") or "").strip(),
                    # O ground truth factual e derivado dos trechos-fonte.
                    ground_truth_facts_text="\n".join(t for t in (trecho_1, trecho_2) if t),
                )
            )
    if not rows:
        raise ValueError("Nenhuma linha valida no CSV")
    return rows


qa_rows = load_qa_estruturado_csv(QA_CSV)
print(f"Perguntas carregadas: {len(qa_rows)}")

Perguntas carregadas: 12


## 1) Indexacao no OpenSearch (Naive RAG)

Antes de comparar, execute esta secao para criar/atualizar o indice vetorial no OpenSearch.

In [55]:
from naive_rag_opensearch.rag import index_documents, retrieve, answer_question
from opensearchpy.exceptions import ConnectionError as OpenSearchConnectionError

try:
    indexed = index_documents(DOCS_DIR)
except OpenSearchConnectionError as exc:
    raise RuntimeError(
        "Falha ao conectar no OpenSearch. Verifique se o container esta no ar e se "
        "OPENSEARCH_HOST/OPENSEARCH_PORT correspondem ao ambiente atual "
        "(dentro do compose: opensearch:9200; fora: localhost:9300)."
    ) from exc

print(f"Chunks indexados no OpenSearch: {indexed}")

# Smoke test de busca
preview = retrieve("Qual e a data do incidente?", k=3)
print(f"Top-3 recuperados: {len(preview)}")
for i, d in enumerate(preview, start=1):
    print(f"[{i}] {d.get('source')} | {d.get('chunk_id')} | score={d.get('score')}")

Chunks indexados no OpenSearch: 39
Top-3 recuperados: 3
[1] Accurate_Energetic_Solutions_Investigation_Update_Publication1.txt | Accurate_Energetic_Solutions_Investigation_Update_Publication1.txt::chunk-0 | score=0.47348982
[2] Accurate_Energetic_Solutions_Investigation_Update_Publication1.txt | Accurate_Energetic_Solutions_Investigation_Update_Publication1.txt::chunk-1 | score=0.47194734
[3] Accurate_Energetic_Solutions_Investigation_Update_Publication1.txt | Accurate_Energetic_Solutions_Investigation_Update_Publication1.txt::chunk-2 | score=0.467656


## 2) Funcoes GraphRAG e Naive RAG (OpenSearch)

In [56]:
def run_graphrag(question: str, thread_id: str) -> dict:
    import graphrag.config as graphrag_config
    import graphrag.graph.nodes as graph_nodes
    import graphrag.graph.query_graph as query_graph

    # Recarrega para refletir alteracoes recentes sem reiniciar kernel.
    # Tambem recarrega config para captar override de modelo por variavel de ambiente.
    importlib.reload(graphrag_config)
    importlib.reload(graph_nodes)
    importlib.reload(query_graph)

    compiled = query_graph.get_compiled_graph()
    cfg = {"configurable": {"thread_id": f"nb-v2-{thread_id}"}}
    try:
        return compiled.invoke({"question": question}, config=cfg)
    except Exception as exc:
        return {
            "final_answer": "",
            "context_docs": [],
            "cypher_result": "",
            "cypher_error": f"{type(exc).__name__}: {exc}",
        }


def graphrag_retrieved_context_strings(state: dict) -> list[str]:
    """Cada elemento e um bloco do contexto entregue ao LLM (entidades, relacionamentos, trechos, etc.)."""
    out: list[str] = []
    for doc in state.get("context_docs") or []:
        if not isinstance(doc, dict):
            continue
        txt = (doc.get("page_content") or "").strip()
        if txt:
            out.append(txt)
    return out


def naive_opensearch_top_k(question: str, k: int) -> tuple[list[str], list[str], str]:
    # Garante funcionamento mesmo se a celula de indexacao/import nao tiver sido executada no kernel atual.
    _retrieve = globals().get("retrieve")

    if _retrieve is None:
        try:
            from naive_rag_opensearch.rag import retrieve as _retrieve
        except Exception as exc:
            raise RuntimeError(
                "Funcao retrieve do Naive RAG OpenSearch indisponivel. "
                "Execute a celula de indexacao/import (secao 1) antes do benchmark."
            ) from exc

    hits = _retrieve(question, k=k)
    ids = [str(h.get("chunk_id") or "") for h in hits]
    texts = [str(h.get("text") or "") for h in hits]

    answer_with_llm = globals().get("_answer_with_benchmark_llm")
    if callable(answer_with_llm):
        answer = str(answer_with_llm(question, texts)).strip()
    else:
        _answer_question = globals().get("answer_question")
        if _answer_question is None:
            from naive_rag_opensearch.rag import answer_question as _answer_question
        rag_out = _answer_question(question, k=k)
        answer = (rag_out.get("answer") or "").strip()

    return ids, texts, answer

## 3) Executar comparacao (Naive OpenSearch vs GraphRAG)

In [57]:
import nest_asyncio
from openai import AsyncOpenAI, AsyncAzureOpenAI, OpenAI, AzureOpenAI

nest_asyncio.apply()

from ragas.llms import llm_factory
from ragas.embeddings import embedding_factory
from ragas.metrics.collections import AnswerRelevancy


_benchmark_llm_provider = os.getenv("BENCHMARK_LLM_PROVIDER", "openai").strip().lower()
_benchmark_llm_model = os.getenv(
    "BENCHMARK_LLM_MODEL",
    os.getenv("GRAPHRAG_LLM_MODEL", os.getenv("OPENAI_CHAT_MODEL", "gpt-4.1-2025-04-14")),
).strip()

if not _benchmark_llm_model:
    raise RuntimeError("BENCHMARK_LLM_MODEL vazio. Configure um modelo valido.")

# Um unico modelo para as abordagens (GraphRAG/Naive) e para as avaliacoes (RAGAS).
os.environ["GRAPHRAG_LLM_MODEL"] = _benchmark_llm_model
os.environ["OPENAI_CHAT_MODEL"] = _benchmark_llm_model
if not os.getenv("RAGAS_LLM_MODEL"):
    os.environ["RAGAS_LLM_MODEL"] = _benchmark_llm_model

_ragas_model = os.getenv("RAGAS_LLM_MODEL", _benchmark_llm_model)
_ragas_embeddings_provider = os.getenv("RAGAS_EMBEDDINGS_PROVIDER", "openai")
_ragas_embeddings_model = os.getenv("RAGAS_EMBEDDINGS_MODEL", "text-embedding-3-small")


def _build_ragas_async_client() -> AsyncOpenAI | AsyncAzureOpenAI:
    if _benchmark_llm_provider == "azure":
        azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT", "").strip()
        azure_api_version = os.getenv("AZURE_OPENAI_API_VERSION", "2024-10-21").strip()
        azure_api_key = os.getenv("AZURE_OPENAI_API_KEY") or os.getenv("OPENAI_API_KEY")
        if not azure_endpoint:
            raise RuntimeError("AZURE_OPENAI_ENDPOINT nao definido para BENCHMARK_LLM_PROVIDER=azure")
        if not azure_api_key:
            raise RuntimeError("AZURE_OPENAI_API_KEY (ou OPENAI_API_KEY) nao definido para Azure")
        return AsyncAzureOpenAI(
            api_key=azure_api_key,
            azure_endpoint=azure_endpoint,
            api_version=azure_api_version,
        )

    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError("OPENAI_API_KEY nao definido para BENCHMARK_LLM_PROVIDER=openai")
    return AsyncOpenAI(api_key=os.environ["OPENAI_API_KEY"])


def _build_sync_chat_client() -> OpenAI | AzureOpenAI:
    if _benchmark_llm_provider == "azure":
        azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT", "").strip()
        azure_api_version = os.getenv("AZURE_OPENAI_API_VERSION", "2024-10-21").strip()
        azure_api_key = os.getenv("AZURE_OPENAI_API_KEY") or os.getenv("OPENAI_API_KEY")
        if not azure_endpoint:
            raise RuntimeError("AZURE_OPENAI_ENDPOINT nao definido para BENCHMARK_LLM_PROVIDER=azure")
        if not azure_api_key:
            raise RuntimeError("AZURE_OPENAI_API_KEY (ou OPENAI_API_KEY) nao definido para Azure")
        return AzureOpenAI(
            api_key=azure_api_key,
            azure_endpoint=azure_endpoint,
            api_version=azure_api_version,
        )

    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError("OPENAI_API_KEY nao definido para BENCHMARK_LLM_PROVIDER=openai")
    return OpenAI(api_key=os.environ["OPENAI_API_KEY"])


_ragas_client = _build_ragas_async_client()
_ragas_llm = llm_factory(_ragas_model, client=_ragas_client)

# Compatibilidade entre versoes do RAGAS:
# - algumas esperam embedding_factory(provider, model=...)
# - outras podem aceitar apenas provider
try:
    _ragas_embeddings = embedding_factory(
        _ragas_embeddings_provider,
        model=_ragas_embeddings_model,
        client=_ragas_client,
    )
except TypeError:
    _ragas_embeddings = embedding_factory(_ragas_embeddings_provider)


_metric_answer_relevancy = AnswerRelevancy(llm=_ragas_llm, embeddings=_ragas_embeddings)


def _answer_with_benchmark_llm(question: str, retrieved_contexts: list[str]) -> str:
    contexts_payload = "\n\n".join(
        f"[{index}] {context_block.strip()}"
        for index, context_block in enumerate(retrieved_contexts)
        if (context_block or "").strip()
    )
    if not contexts_payload:
        return "Nao encontrei contexto relevante para responder com seguranca."

    prompt = (
        "Responda em portugues usando somente os contextos recuperados. "
        "Se faltar evidencia suficiente, diga explicitamente que nao encontrou no contexto.\n\n"
        f"Contextos:\n{contexts_payload}\n\n"
        f"Pergunta: {question}"
    )
    completion = _build_sync_chat_client().chat.completions.create(
        model=_benchmark_llm_model,
        temperature=0,
        messages=[{"role": "user", "content": prompt}],
    )
    return (completion.choices[0].message.content or "").strip()


print(f"BENCHMARK: provider LLM = {_benchmark_llm_provider}")
print(f"BENCHMARK: modelo LLM = {_benchmark_llm_model}")
print(f"RAGAS: modelo de juizo = {_ragas_model}")
print(f"RAGAS: provider de embeddings = {_ragas_embeddings_provider}")
print(f"RAGAS: modelo de embeddings = {_ragas_embeddings_model}")

_ragas_max_ctx = int(os.getenv("RAGAS_MAX_CONTEXT_BLOCKS", "16"))
print(f"RAGAS: max blocos de contexto por pergunta = {_ragas_max_ctx}")


def _as_float_metric(x) -> float:
    try:
        v = float(x)
    except (TypeError, ValueError):
        return float("nan")
    return v if not math.isnan(v) else float("nan")


def _build_structured_eval_llm():
    from graphrag.llm_factory import create_chat_llm

    return create_chat_llm(model=_ragas_model, temperature=0)


async def ragas_context_precision(
    question: str,
    retrieved_contexts: list[str],
) -> float:
    """Estimativa de Context Precision sem ground-truth.

    Mede a proporcao de blocos de contexto que sao relevantes para responder a pergunta.
    """
    question_text = (question or "").strip()
    retrieved_context_blocks = [
        context_block.strip()
        for context_block in retrieved_contexts
        if (context_block or "").strip()
    ][:_ragas_max_ctx]

    if not question_text or not retrieved_context_blocks:
        return float("nan")

    from pydantic import BaseModel, Field

    class ContextRelevanceItem(BaseModel):
        index: int = Field(..., ge=0)
        relevant: bool

    class ContextRelevanceAssessment(BaseModel):
        items: list[ContextRelevanceItem] = Field(default_factory=list)

    contexts_payload = "\n\n".join(
        f"[{index}] {context_block}"
        for index, context_block in enumerate(retrieved_context_blocks)
    )

    evaluation_prompt = (
        "Voce e um avaliador de recuperacao de informacao. "
        "Para cada bloco de contexto, marque relevant=true apenas se ele contribuir "
        "diretamente para responder a pergunta. "
        "Se o bloco for irrelevante, redundante sem adicionar informacao util, "
        "ou muito generico, marque relevant=false."
    )

    try:
        llm = _build_structured_eval_llm()
        structured_llm = llm.with_structured_output(ContextRelevanceAssessment)
        assessment = await structured_llm.ainvoke(
            [
                ("system", evaluation_prompt),
                (
                    "human",
                    f"Pergunta:\n{question_text}\n\nBlocos de contexto (indexados):\n{contexts_payload}",
                ),
            ]
        )

        relevant_indexes = {
            item.index
            for item in (assessment.items if assessment else [])
            if item.relevant and 0 <= item.index < len(retrieved_context_blocks)
        }

        return len(relevant_indexes) / len(retrieved_context_blocks)
    except Exception as exc:
        print(f"Context Precision (aviso): {type(exc).__name__}: {exc}")
        return float("nan")


async def ragas_context_recall(
    question: str,
    retrieved_contexts: list[str],
    expected_answer: str,
    generated_answer: str,
) -> float:
    """Estimativa manual de Context Recall com avaliacao estruturada por LLM.

    Mede quanto da resposta esperada foi coberto pela resposta gerada,
    considerando apenas fatos suportados pelos contextos recuperados.
    """
    question_text = (question or "").strip()
    expected_answer_text = (expected_answer or "").strip()
    generated_answer_text = (generated_answer or "").strip()
    retrieved_context_blocks = [
        context_block.strip()
        for context_block in retrieved_contexts
        if (context_block or "").strip()
    ][:_ragas_max_ctx]

    if not question_text or not expected_answer_text or not generated_answer_text or not retrieved_context_blocks:
        return float("nan")

    from pydantic import BaseModel, Field

    class ExpectedFactCoverageItem(BaseModel):
        expected_fact: str = Field(..., description="Fato atomico derivado da resposta esperada")
        supported_by_context: bool = Field(..., description="Se os contextos recuperados suportam esse fato")
        covered_in_generated_answer: bool = Field(..., description="Se a resposta gerada cobre corretamente esse fato")

    class ContextRecallAssessment(BaseModel):
        items: list[ExpectedFactCoverageItem] = Field(default_factory=list)

    contexts_payload = "\n\n".join(
        f"[{index}] {context_block}"
        for index, context_block in enumerate(retrieved_context_blocks)
    )

    evaluation_prompt = (
        "Voce e um avaliador de Context Recall em RAG. "
        "Quebre a resposta esperada em fatos atomicos. Para cada fato, indique: "
        "(1) se o fato esta suportado pelos contextos recuperados e "
        "(2) se o fato foi coberto corretamente na resposta gerada. "
        "Considere covered_in_generated_answer=true apenas quando o fato estiver correto e "
        "nao contradizer a resposta esperada."
    )

    try:
        llm = _build_structured_eval_llm()
        structured_llm = llm.with_structured_output(ContextRecallAssessment)
        assessment = await structured_llm.ainvoke(
            [
                ("system", evaluation_prompt),
                (
                    "human",
                    "\n\n".join(
                        [
                            f"Pergunta:\n{question_text}",
                            f"Resposta esperada:\n{expected_answer_text}",
                            f"Resposta gerada:\n{generated_answer_text}",
                            f"Contextos recuperados (indexados):\n{contexts_payload}",
                        ]
                    ),
                ),
            ]
        )

        assessed_items = assessment.items if assessment else []
        supported_facts = [item for item in assessed_items if item.supported_by_context]
        if not supported_facts:
            return 0.0

        covered_supported_facts = [
            item
            for item in supported_facts
            if item.covered_in_generated_answer
        ]
        return len(covered_supported_facts) / len(supported_facts)
    except Exception as exc:
        print(f"Context Recall (aviso): {type(exc).__name__}: {exc}")
        return float("nan")


async def ragas_answer_relevancy(question: str, answer: str) -> float:
    """Retorna a metrica Answer Relevancy do RAGAS; NaN se faltar resposta."""
    ans = (answer or "").strip()
    if not ans:
        return float("nan")
    try:
        ar = await _metric_answer_relevancy.ascore(user_input=question, response=ans)
        return _as_float_metric(ar.value)
    except Exception as exc:
        print(f"RAGAS (aviso): {type(exc).__name__}: {exc}")
        return float("nan")


/tmp/ipykernel_19059/3838031127.py:22: DeprecationWarning: Importing embedding_factory from ragas.embeddings is deprecated. Import directly from ragas.embeddings.base or use modern providers: from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  _ragas_embeddings = embedding_factory(


RAGAS: modelo de juizo = gpt-4.1-2025-04-14
RAGAS: provider de embeddings = openai
RAGAS: modelo de embeddings = text-embedding-3-small
RAGAS: max blocos de contexto por pergunta = 16


In [58]:
K = 5


def _graphrag_cypher_context_strings(graphrag_state: dict) -> list[str]:
    """Retorna somente os contextos vindos da resposta de consulta Cypher."""
    if not isinstance(graphrag_state, dict):
        return []

    cypher_result = graphrag_state.get("cypher_result")
    if cypher_result is None:
        return []

    if isinstance(cypher_result, list):
        return [str(item).strip() for item in cypher_result if str(item).strip()]

    cypher_result_text = str(cypher_result).strip()
    return [cypher_result_text] if cypher_result_text else []


async def _run_benchmark_comparison() -> list[dict]:
    benchmark_results: list[dict] = []
    total_questions = len(qa_rows)

    for question_index, qa_row in enumerate(qa_rows, start=1):
        thread_tag = str(abs(hash(qa_row.question)) % 10_000_000)

        naive_retrieved_ids, naive_retrieved_texts, naive_generated_answer = naive_opensearch_top_k(
            qa_row.question,
            K,
        )
        naive_context_precision = await ragas_context_precision(
            question=qa_row.question,
            retrieved_contexts=naive_retrieved_texts,
        )
        naive_context_recall = await ragas_context_recall(
            question=qa_row.question,
            retrieved_contexts=naive_retrieved_texts,
            expected_answer=qa_row.reference_answer,
            generated_answer=naive_generated_answer,
        )
        naive_answer_relevancy = await ragas_answer_relevancy(
            qa_row.question,
            naive_generated_answer,
        )

        graphrag_state = run_graphrag(qa_row.question, thread_tag)
        graphrag_generated_answer = (
            (graphrag_state.get("final_answer") or "").strip()
            if isinstance(graphrag_state, dict)
            else ""
        )
        graphrag_cypher_contexts = _graphrag_cypher_context_strings(graphrag_state)
        graphrag_context_precision = await ragas_context_precision(
            question=qa_row.question,
            retrieved_contexts=graphrag_cypher_contexts,
        )
        graphrag_context_recall = await ragas_context_recall(
            question=qa_row.question,
            retrieved_contexts=graphrag_cypher_contexts,
            expected_answer=qa_row.reference_answer,
            generated_answer=graphrag_generated_answer,
        )
        graphrag_answer_relevancy = await ragas_answer_relevancy(
            qa_row.question,
            graphrag_generated_answer,
        )

        graphrag_error = (
            (graphrag_state.get("cypher_error") or "").strip()
            if isinstance(graphrag_state, dict)
            else ""
        )
        if graphrag_error:
            print(f"[{question_index}/{total_questions}] GraphRAG fallback: {graphrag_error[:160]}")

        benchmark_results.append(
            {
                "qid": qa_row.qid,
                "q": question_index,
                "hop": qa_row.tipo_pergunta,
                "system": "naive_rag_opensearch",
                "context_precision": naive_context_precision,
                "context_recall": naive_context_recall,
                "answer_relevancy": naive_answer_relevancy,
                "question": qa_row.question,
                "reference": qa_row.reference_answer,
                "contexts_retrieved": naive_retrieved_texts,
                "generated": naive_generated_answer,
            }
        )
        benchmark_results.append(
            {
                "qid": qa_row.qid,
                "q": question_index,
                "hop": qa_row.tipo_pergunta,
                "system": "graphrag",
                "context_precision": graphrag_context_precision,
                "context_recall": graphrag_context_recall,
                "answer_relevancy": graphrag_answer_relevancy,
                "question": qa_row.question,
                "reference": qa_row.reference_answer,
                "contexts_retrieved": graphrag_cypher_contexts,
                "generated": graphrag_generated_answer,
            }
        )

        naive_context_precision_log = (
            naive_context_precision if not math.isnan(naive_context_precision) else -1.0
        )
        graphrag_context_precision_log = (
            graphrag_context_precision if not math.isnan(graphrag_context_precision) else -1.0
        )
        naive_context_recall_log = (
            naive_context_recall if not math.isnan(naive_context_recall) else -1.0
        )
        graphrag_context_recall_log = (
            graphrag_context_recall if not math.isnan(graphrag_context_recall) else -1.0
        )
        print(
            f"[{question_index}/{total_questions}] {qa_row.tipo_pergunta} "
            f"naive_cp={naive_context_precision_log:.3f} "
            f"graphrag_cp={graphrag_context_precision_log:.3f} "
            f"naive_cr={naive_context_recall_log:.3f} "
            f"graphrag_cr={graphrag_context_recall_log:.3f} "
            f"naive_as={naive_answer_relevancy:.3f} "
            f"graphrag_as={graphrag_answer_relevancy:.3f}"
        )

    print("Concluido.")
    return benchmark_results


results = await _run_benchmark_comparison()

Decision: local
Subqueries: [SubQuery(sub_query='Qual foi a data do incidente de explosão na Accurate Energetic Systems?'), SubQuery(sub_query='Explosão na Accurate Energetic Systems: quando ocorreu o incidente?')]
Local retrieve: 6 seed entities, 26 context chunks
Graph QA Cypher (attempt 1): MATCH (e:Entity {name: "explosion in Building 602"})-[:HAS_CLAIM]->(c:Claim)
WHERE c.text CONTAINS "7:47 a.m." OR c.text CONTAINS "October 2025"
RETURN c.text
LIMIT 25
[1/12] single-hop naive_cp=0.600 graphrag_cp=0.000 naive_cr=1.000 graphrag_cr=0.000 naive_as=0.992 graphrag_as=0.951
Decision: local
Subqueries: [SubQuery(sub_query='Encontrar o número de funcionários que morreram no incidente descrito.'), SubQuery(sub_query='Identificar o incidente mencionado e listar os funcionários afetados e seus desfechos.')]
Local retrieve: 8 seed entities, 27 context chunks
Graph QA Cypher (attempt 1): MATCH (e:Entity {name: "employees"})-[:HAS_COVARIATE]->(c:Covariate)
WHERE toLower(c.name) CONTAINS "fatal"

In [59]:
results

[{'qid': 'Q01',
  'q': 1,
  'hop': 'single-hop',
  'system': 'naive_rag_opensearch',
  'context_precision': 0.6,
  'context_recall': 1.0,
  'answer_relevancy': 0.9923677838643435,
  'question': 'Qual é a data do incidente na explosão da Accurate Energetic Systems?',
  'reference': 'October 10, 2025.',
  'contexts_retrieved': ['**Fatal Explosion at Accurate Energetic Systems McEwen, Tennessee** | _Incident Date: October 10, 2025_ | **No. 2025-TN-I-04** \n\n**==> picture [389 x 16] intentionally omitted <==**\n\n**==> picture [84 x 39] intentionally omitted <==**\n\n**U.S. Chemical Safety and Hazard Investigation Board** \n\n**==> picture [526 x 6] intentionally omitted <==**\n\n## **Investigation Update** \n\n## March 2026 \n\nThis document provides an update on the CSB’s investigation of the October 10, 2025, incident at Accurate Energetic Systems, LLC. \n\n## **Incident Summary** \n\nOn October 10, 2025, at 7:47 a.m., multiple catastrophic explosions fatally injured 16 employees at th

In [60]:
try:
    import pandas as pd
except ImportError:
    pd = None

if pd is not None:
    results_dataframe = pd.DataFrame(results)
    display_columns = [
        "q",
        "qid",
        "hop",
        "system",
        "question",
        "reference",
        "generated",
        "context_precision",
        "context_recall",
        "answer_relevancy",
    ]
    display(results_dataframe[display_columns])

    metrics_summary = (
        results_dataframe.groupby("system")[["context_precision", "context_recall", "answer_relevancy"]]
        .mean(numeric_only=True)
        .rename_axis("media macro")
    )
    display(metrics_summary)
else:
    from statistics import mean

    for system_name in sorted(set(result_row["system"] for result_row in results)):
        system_rows = [result_row for result_row in results if result_row["system"] == system_name]

        def mean_metric(metric_name: str) -> float:
            metric_values = [
                row[metric_name]
                for row in system_rows
                if isinstance(row[metric_name], (int, float)) and not math.isnan(row[metric_name])
            ]
            return mean(metric_values) if metric_values else float("nan")

        print(
            system_name,
            "context_recall",
            mean_metric("context_recall"),
            "context_precision",
            mean_metric("context_precision"),
            "answer_relevancy",
            mean_metric("answer_relevancy"),
        )

,q,qid,hop,system,question,reference,generated,context_precision,context_recall,answer_relevancy
0,1,Q01,single-hop,naive_rag_opensearch,Qual é a data do incidente na explosão da Accu...,"October 10, 2025.",A data do incidente na explosão da Accurate En...,0.6,1.0,0.992368
1,1,Q01,single-hop,graphrag,Qual é a data do incidente na explosão da Accu...,"October 10, 2025.",O incidente de explosão na Accurate Energetic ...,0.0,0.0,0.950917
2,2,Q02,single-hop,naive_rag_opensearch,Quantos funcionários morreram no incidente des...,16 funcionários.,"De acordo com o contexto fornecido, 16 funcion...",0.4,1.0,1.000000
3,2,Q02,single-hop,graphrag,Quantos funcionários morreram no incidente des...,16 funcionários.,"De acordo com o contexto fornecido, 16 funcion...",0.0,0.0,1.000000
4,3,Q03,single-hop,naive_rag_opensearch,Qual era o nome do edifício onde ocorreu o inc...,Building 602.,O nome do edifício onde ocorreu o incidente er...,0.8,1.0,1.000000
5,3,Q03,single-hop,graphrag,Qual era o nome do edifício onde ocorreu o inc...,Building 602.,O nome do edifício onde ocorreu o incidente er...,1.0,1.0,1.000000
6,4,Q04,multi-hop,naive_rag_opensearch,Quantos quilos de explosivos estavam presentes...,Aproximadamente 24.600 libras estavam presente...,"No momento do incidente, aproximadamente 24.60...",0.4,1.0,0.969648
7,4,Q04,multi-hop,graphrag,Quantos quilos de explosivos estavam presentes...,Aproximadamente 24.600 libras estavam presente...,"No momento do incidente, havia aproximadamente...",1.0,1.0,0.969648
8,5,Q05,multi-hop,naive_rag_opensearch,Quais materiais explosivos foram usados na com...,Comp B é composto por 60% RDX e 40% TNT.,Comp B é uma mistura composta por 60% de RDX e...,0.4,1.0,0.616039
9,5,Q05,multi-hop,graphrag,Quais materiais explosivos foram usados na com...,Comp B é composto por 60% RDX e 40% TNT.,O Comp B é composto por dois materiais explosi...,1.0,1.0,0.870298


,context_precision,context_recall,answer_relevancy
media macro,,,
graphrag,0.636364,0.545455,0.882743
naive_rag_opensearch,0.450000,0.916667,0.819247


In [61]:
try:
    import pandas as pd
except ImportError:
    pd = None


def _normaliza_categoria(hop: str) -> str:
    h = (hop or "").strip().lower().replace("_", "-")
    if "single" in h:
        return "single-hop"
    if "multi" in h:
        return "multi-hop"
    if "global" in h:
        return "global"
    if "tabela" in h:
        return "tabela"
    return h


categorias_alvo = ["single-hop", "multi-hop", "global", "tabela"]
abordagens_alvo = ["graphrag", "naive_rag_opensearch"]
metricas_alvo = ["context_precision", "context_recall", "answer_relevancy"]

if pd is not None:
    df = pd.DataFrame(results).copy()
    df["categoria"] = df["hop"].map(_normaliza_categoria)

    base = df[df["categoria"].isin(categorias_alvo) & df["system"].isin(abordagens_alvo)]

    tabela_metricas = (
        base.groupby(["categoria", "system"], as_index=False)[metricas_alvo]
        .mean(numeric_only=True)
        .set_index(["categoria", "system"])
        .unstack("system")
        .reindex(categorias_alvo)
    )

    # Ordena MultiIndex de colunas para (metrica, abordagem).
    tabela_metricas = tabela_metricas.reindex(
        columns=pd.MultiIndex.from_product([metricas_alvo, abordagens_alvo])
    )

    display(tabela_metricas)
else:
    from statistics import mean

    grouped_metric_values = {
        (category_name, system_name, metric_name): []
        for category_name in categorias_alvo
        for system_name in abordagens_alvo
        for metric_name in metricas_alvo
    }

    for result_row in results:
        category_name = _normaliza_categoria(result_row.get("hop", ""))
        system_name = result_row.get("system", "")
        if category_name not in categorias_alvo or system_name not in abordagens_alvo:
            continue
        for metric_name in metricas_alvo:
            metric_value = result_row.get(metric_name)
            if isinstance(metric_value, (int, float)) and not math.isnan(metric_value):
                grouped_metric_values[(category_name, system_name, metric_name)].append(metric_value)

    for category_name in categorias_alvo:
        print(f"\nCategoria: {category_name}")
        for metric_name in metricas_alvo:
            print(f"  {metric_name}:")
            for system_name in abordagens_alvo:
                metric_values = grouped_metric_values[(category_name, system_name, metric_name)]
                metric_mean = mean(metric_values) if metric_values else float("nan")
                print(f"    {system_name}: {metric_mean}")

context_precision                      context_recall  \
                    graphrag naive_rag_opensearch       graphrag   
categoria                                                          
single-hop          0.333333             0.600000       0.333333   
multi-hop           1.000000             0.466667       1.000000   
global              1.000000             0.533333       0.500000   
tabela              0.333333             0.200000       0.333333   

                                answer_relevancy                       
           naive_rag_opensearch         graphrag naive_rag_opensearch  
categoria                                                              
single-hop             1.000000         0.983639             0.997456  
multi-hop              1.000000         0.814781             0.754552  
global                 0.666667         0.814179             0.575967  
tabela                 1.000000         0.918374             0.949014